# Auto Calibration — QickworkspaceV2

This notebook demonstrates the full automated single-qubit ge calibration pipeline.

| Cell | What it covers |
| --- | --- |
| 1 | Imports & setup |
| 2 | `CalibrationStore` — load / inspect |
| 3 | `CalibrationGraph` — visualise the DAG |
| 4 | `CalibrationMonitor` — background staleness watcher |
| 5 | `AutoCalibrate` — full 7-step pipeline |
| 6 | Results dashboard |


---
## 1 — Imports & hardware setup

In [ ]:
import sys, os
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'QickworkspaceV2').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from datetime import datetime

from QickworkspaceV2 import (
    ExperimentConfig,
    CalibrationStore, CalibrationGraph, CalibrationNode, CalibrationMonitor,
    AutoCalibrate,
)
from QickworkspaceV2.core.base_experiment import BaseExperiment
from QickworkspaceV2.config.system_cfg import config_list

print('QickworkspaceV2 loaded')


In [ ]:
# Real QICK hardware
from QickworkspaceV2 import BaseExperiment

BaseExperiment.connect_pyro4('192.168.10.82', data_path='D:/Labber_Data/Jay/purcell_tmon/Rshield/temperature')

# Config
qubit      = 'Q1'
config_all = ExperimentConfig(config_list)
run_cfg    = config_all.get_qubit(qubit)

# Centralised initial guesses for auto calibration.
# Tuple format: (start, stop, points). For time axes use us.
init_guess = {
    'res': (run_cfg['res_freq_ge'] - 10, run_cfg['res_freq_ge'] + 10, 101),
    'qb':  (run_cfg['qb_freq_ge'] - 50,  run_cfg['qb_freq_ge'] + 50,  101),
    'rabi': (0.0, 1.0, 100),
    'sigma': run_cfg.get('sigma_ge', 0.01),
    'T1_guess': 50.0,
    'T2_guess': 40.0,
    't1': (0.0, 150.0, 100),
    'ramsey': (0.0, 80.0, 100),
    'spin_echo': (0.0, 120.0, 100),
    'virtual_detune': 2.0,
}

store = CalibrationStore(f'cal_{qubit}.json')
print('QICK session active')
print(f'Qubit: {qubit}')


---
## 2 — CalibrationStore: inspect current state

In [ ]:
def show_store(store: CalibrationStore, qubit: str, max_age_hours: float = 24):
    """Print all stored parameters with age and freshness."""
    keys = sorted(store.all_keys(qubit))
    if not keys:
        print(f'  [{qubit}] store is empty - run calibration first.')
        return

    now = datetime.now()
    print(f'\n  Calibration Store  -  {qubit}')
    print(f'  {"Key":<28s}  {"Value":<18s}  {"Age":<14s}  Status')
    print('  ' + '-' * 78)
    for key in keys:
        val = store.get(qubit, key)
        ts = store.timestamp(qubit, key)
        age_h = (now - ts).total_seconds() / 3600 if ts else float('inf')
        fresh = 'fresh' if not store.is_stale(qubit, key, max_age_hours=max_age_hours) else 'stale'
        val_str = f'{val:.5g}' if isinstance(val, float) else str(val)
        age_str = f'{age_h:.1f}h' if ts else 'missing ts'
        print(f'  {key:<28s}  {val_str:<18s}  {age_str:<14s}  {fresh}')

show_store(store, qubit)


---
## 3 — CalibrationGraph: DAG visualisation

In [ ]:
PIPELINE_NODES = [
    CalibrationNode('res_spec',   lambda: None,
                    provides=['res_freq_ge'],
                    requires=[]),
    CalibrationNode('qubit_spec', lambda: None,
                    provides=['qb_freq_ge', 'qb_mixer'],
                    requires=['res_freq_ge']),
    CalibrationNode('power_rabi', lambda: None,
                    provides=['pi_gain_ge', 'pi2_gain_ge'],
                    requires=['qb_freq_ge']),
    CalibrationNode('t1',         lambda: None,
                    provides=['T1_us'],
                    requires=['pi_gain_ge']),
    CalibrationNode('ramsey',     lambda: None,
                    provides=['qb_freq_ge_corrected', 'T2r_us'],
                    requires=['pi_gain_ge', 'qb_freq_ge', 'T1_us']),
    CalibrationNode('spin_echo',  lambda: None,
                    provides=['T2e_us'],
                    requires=['pi_gain_ge', 'T2r_us', 'T1_us']),
    CalibrationNode('ss_opt',     lambda: None,
                    provides=['ro_length', 'res_gain_ge', 'res_freq_ge_opt', 'readout_fidelity'],
                    requires=['pi_gain_ge', 'qb_freq_ge_corrected']),
]

graph = CalibrationGraph(store, qubit)
for node in PIPELINE_NODES:
    graph.add(node)

print(graph.status())


In [ ]:
def plot_calibration_graph(nodes: list[CalibrationNode], store: CalibrationStore, qubit: str,
                           max_age_hours: float = 24, figsize=(13, 6)):
    """Draw the calibration DAG with staleness colour coding."""
    G = nx.DiGraph()
    node_by_name = {node.name: node for node in nodes}

    for node in nodes:
        G.add_node(node.name)

    provided_by = {}
    for node in nodes:
        for key in node.provides:
            provided_by[key] = node.name

    for node in nodes:
        for req in node.requires:
            provider = provided_by.get(req)
            if provider and provider != node.name:
                G.add_edge(provider, node.name, label=req)

    node_colors = []
    for name in G.nodes():
        node = node_by_name[name]
        stale = any(store.is_stale(qubit, key, max_age_hours=max_age_hours) for key in node.provides)
        node_colors.append('#FF6B6B' if stale else '#6BCB77')

    try:
        pos = nx.nx_agraph.graphviz_layout(G, prog='dot')
    except Exception:
        pos = nx.spring_layout(G, seed=42, k=2.5)

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_title(f'Calibration DAG  -  {qubit}', fontsize=14, fontweight='bold', pad=16)

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=2800, alpha=0.92)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=10, font_weight='bold')
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#555', arrows=True,
                           arrowsize=20, arrowstyle='-|>', width=1.8,
                           connectionstyle='arc3,rad=0.07',
                           min_source_margin=25, min_target_margin=25)

    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax,
                                 font_size=8, font_color='#333',
                                 bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

    legend_handles = [
        mpatches.Patch(color='#6BCB77', label=f'Fresh (< {max_age_hours:g} h)'),
        mpatches.Patch(color='#FF6B6B', label='Stale / not yet run'),
    ]
    ax.legend(handles=legend_handles, loc='upper right', framealpha=0.85)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

plot_calibration_graph(PIPELINE_NODES, store, qubit)


---
## 4 — CalibrationMonitor: background staleness watcher

In [ ]:
# Optional background staleness watcher. Set RUN_MONITOR=True when you want it.
RUN_MONITOR = False
stale_log = []

def on_stale(qubit_label: str, key: str):
    ts = datetime.now().strftime('%H:%M:%S')
    msg = f'[{ts}] STALE - {qubit_label}/{key}'
    stale_log.append((ts, f'{qubit_label}/{key}'))
    print(msg)

monitor = CalibrationMonitor(
    store,
    poll_interval_s=60,
    alert_fn=on_stale,
    max_age_hours=24,
)
monitor.watch(qubit)

if RUN_MONITOR:
    monitor.start()
    print('Monitor started (polling every 60 s).')
    print('Call monitor.stop() to shut it down.')
else:
    print('Monitor configured but not started. Set RUN_MONITOR=True to enable polling.')


In [ ]:
# Inspect stale log (run any time)
if stale_log:
    print(f'Stale events so far ({len(stale_log)}):')
    for ts, key in stale_log:
        print(f'  {ts}  {key}')
else:
    print('No stale events yet.')

In [ ]:
# Stop monitor when done.
if 'monitor' in globals() and monitor.running():
    monitor.stop()
    print('Monitor stopped.')
else:
    print('Monitor is not running.')


---
## 5 — AutoCalibrate: full 7-step pipeline

In [ ]:
# Run the standard auto-calibration steps.
auto = AutoCalibrate(config_all, qubit, cal_store=store, init_guess=init_guess)

auto.run(
    skip=(
        # 'res_spec',    # uncomment to skip individual steps
        # 'qubit_spec',
        # 'power_rabi',
        # 'ramsey',
        # 'spin_echo',
        # 't1',
        'ss_opt',      # skip readout optimisation (needs many shots)
    )
)


In [ ]:
# Run individual steps manually (useful for re-running a single step)
# auto.step_res_spec(span=10, steps=101, py_avg=5)
# auto.step_qubit_spec(span=50, steps=101, py_avg=5)
# auto.step_power_rabi(steps=100, py_avg=10)
# auto.step_ramsey(steps=100, py_avg=20, virtual_detune=2.0)
# auto.step_spin_echo(steps=100, py_avg=100)
# auto.step_t1(steps=100, py_avg=50)
# auto.step_ss_opt(shots=1000)

In [ ]:
auto.summary()

---
## 6 — Results dashboard

In [ ]:
def plot_results_dashboard(results: dict, qubit: str):
    """Bar + text summary of auto-calibration outcomes."""
    if not results:
        print('No results yet - run AutoCalibrate first.')
        return

    coh_keys = {'T1_us': 'T1 (us)', 'T2e_us': 'T2 echo (us)', 'T2r_us': 'T2* (us)'}
    coh_vals = {label: results[k] for k, label in coh_keys.items() if k in results}

    freq_keys = {'res_freq_ge': 'f_res (MHz)', 'qb_freq_ge': 'f_qb (MHz)'}
    freq_vals = {label: results[k] for k, label in freq_keys.items() if k in results}

    gain_keys = {'pi_gain_ge': 'pi gain', 'pi2_gain_ge': 'pi/2 gain'}
    gain_vals = {label: results[k] for k, label in gain_keys.items() if k in results}

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    fig.suptitle(f'Auto-Calibration Results  -  {qubit}', fontsize=14, fontweight='bold', y=1.02)

    panels = [
        (axes[0], coh_vals,  'Coherence Times', '#4C9BE8', 'us'),
        (axes[1], freq_vals, 'Frequencies',     '#F4A261', 'MHz'),
        (axes[2], gain_vals, 'Pulse Gains',     '#6BCB77', ''),
    ]

    for ax, data, title, color, unit in panels:
        ax.set_title(title, fontweight='bold', fontsize=11)
        if not data:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes,
                    fontsize=13, color='grey')
            ax.axis('off')
            continue
        labels = list(data.keys())
        values = list(data.values())
        bars = ax.barh(labels, values, color=color, alpha=0.85, edgecolor='white', height=0.55)
        for bar, val in zip(bars, values):
            label_str = f'{val:.2f} {unit}' if isinstance(val, float) else str(val)
            ax.text(val * 1.01, bar.get_y() + bar.get_height() / 2,
                    label_str, va='center', fontsize=10)
        ax.set_xlabel(unit if unit else 'Value')
        ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.show()

    print(f'\n  Full results  -  {qubit}')
    print(f'  {"Parameter":<28s}  Value')
    print('  ' + '-' * 45)
    for key, value in results.items():
        val_str = f'{value:.6g}' if isinstance(value, float) else str(value)
        print(f'  {key:<28s}  {val_str}')

plot_results_dashboard(auto.results, qubit)


In [ ]:
# Refresh the graph after calibration.
# AutoCalibrate writes successful step outputs to the CalibrationStore.
print(graph.status())
plot_calibration_graph(PIPELINE_NODES, store, qubit)


---
## 7 — CalibrationGraph: run only stale nodes

In [ ]:
# Wire each node's run_fn to the corresponding AutoCalibrate step
# so CalibrationGraph.run_stale() can execute them automatically.

auto2 = AutoCalibrate(config_all, qubit, cal_store=store, init_guess=init_guess)

step_fns = {
    'res_spec':   auto2.step_res_spec,
    'qubit_spec': auto2.step_qubit_spec,
    'power_rabi': auto2.step_power_rabi,
    'ramsey':     auto2.step_ramsey,
    'spin_echo':  auto2.step_spin_echo,
    't1':         auto2.step_t1,
    'ss_opt':     auto2.step_ss_opt,
}

graph2 = CalibrationGraph(store, qubit)
for node in PIPELINE_NODES:
    wired = CalibrationNode(node.name, step_fns[node.name],
                            provides=node.provides, requires=node.requires)
    graph2.add(wired)

# Dry-run: see which nodes are stale without executing them
print('Dry-run — nodes that would execute:')
to_run = graph2.run_stale(max_age_hours=24, dry_run=True)
if not to_run:
    print('  All parameters are fresh — nothing to do.')

# Actual run (uncomment to execute):
# graph2.run_stale(max_age_hours=24)

---
## 8 — Store timeline plot

In [ ]:
def plot_store_timeline(store: CalibrationStore, qubit: str, keys=None,
                        max_age_hours: float = 24, figsize=(14, 5)):
    """Show parameter age in hours as a horizontal bar chart."""
    all_keys = store.all_keys(qubit)
    if keys:
        all_keys = [k for k in all_keys if k in keys]
    if not all_keys:
        print('Store is empty.')
        return

    now = datetime.now()
    ages_h, labels, colors = [], [], []
    for key in all_keys:
        ts = store.timestamp(qubit, key)
        if ts is None:
            continue
        age = (now - ts).total_seconds() / 3600
        ages_h.append(age)
        labels.append(key)
        colors.append('#FF6B6B' if store.is_stale(qubit, key, max_age_hours=max_age_hours) else '#6BCB77')

    if not ages_h:
        print('No timestamped entries found.')
        return

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(labels, ages_h, color=colors, edgecolor='white', height=0.6)
    ax.axvline(max_age_hours, color='tomato', linestyle='--', linewidth=1.5,
               label=f'{max_age_hours:g} h threshold')

    for bar, age in zip(bars, ages_h):
        ax.text(age + 0.1, bar.get_y() + bar.get_height() / 2,
                f'{age:.1f} h', va='center', fontsize=9)

    ax.set_xlabel('Age (hours since last calibration)')
    ax.set_title(f'Parameter Freshness  -  {qubit}', fontweight='bold', fontsize=12)
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

show_store(store, qubit)
plot_store_timeline(store, qubit)


---
## 9 ? Hardware Smoke Test
Run a shortened calibration sequence on the active QICK session.


In [ ]:
# Fresh config & store for a short hardware smoke test
config_test  = ExperimentConfig(config_list)
store_test   = CalibrationStore('cal_smoke.json')
auto_smoke   = AutoCalibrate(config_test, qubit, cal_store=store_test, init_guess=init_guess)

auto_smoke.run(skip=('spin_echo', 'ss_opt'))
auto_smoke.summary()


In [ ]:
plot_results_dashboard(auto_smoke.results, qubit + ' (smoke)')